# Model Training and Evaluation
Notebook ini melatih model Random Forest menggunakan fitur HSI yang telah diekstraksi.

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

train_csv = os.path.join('..', 'data', 'processed', 'train_features.csv')
test_csv = os.path.join('..', 'data', 'processed', 'test_features.csv')

df_train = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)

print("Training data shape:", df_train.shape)
print("Testing data shape:", df_test.shape)

X_train = df_train.drop(['fruit', 'label'], axis=1)
y_train = df_train['label']
X_test = df_test.drop(['fruit', 'label'], axis=1)
y_test = df_test['label']

print("Training model...")
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

print("Evaluating model...")
start_time = time.time()
y_pred = rf.predict(X_test)
end_time = time.time()

latency_per_image = (end_time - start_time) / len(X_test) * 1000
print(f'\nInference Latency: {latency_per_image:.4f} ms per image')

print('\nAccuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=rf.classes_, yticklabels=rf.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.savefig(os.path.join('..', 'reports', 'confusion_matrix.png'))
plt.show()

print("Saving model to models/random_forest.pkl...")
with open(os.path.join('..', 'models', 'random_forest.pkl'), 'wb') as f:
    pickle.dump(rf, f)

print("Model saved successfully!")